# Language Foundation Models

**Prerequisites**

- L10 notebooks 01–03: Tokenization, the transformer architecture, pre-training and post-training
- L11 notebook 01: Foundation model concepts, the four adaptation strategies, LoRA derivation
- L11 notebook 02: Vision foundation models — the load-inspect-adapt pattern

**Outcomes**

- Load pretrained language models (DistilBERT, GPT-2) from Hugging Face `transformers` in three lines
- Understand how encoder models (BERT family) and decoder models (GPT family) are adapted differently
- Fine-tune DistilBERT for text classification using a hand-written PyTorch training loop
- Re-implement LoRA and apply it by hand to GPT-2's attention layers
- Compare parameter counts and training efficiency between full fine-tuning and LoRA
- Understand where fine-tuning sits next to prompting, in-context learning, and RAG

> **Note.** The first time this notebook runs it downloads the DistilBERT (~250 MB) and GPT-2 (~500 MB) checkpoints to `~/.cache/huggingface/`. Subsequent runs are fast.

In [ ]:
import copy
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    GPT2LMHeadModel,
    GPT2Tokenizer,
)
from transformers.pytorch_utils import Conv1D

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## The Hugging Face `transformers` Library

For language models, the standard library is Hugging Face's `transformers`. It provides a uniform interface to load hundreds of pretrained language models — BERT, RoBERTa, DistilBERT, GPT-2, Llama, Mistral, Qwen, and many more — along with their associated tokenizers. The core pattern mirrors what we saw with `torchvision` for vision models: a few lines to load, and then the model is a standard `nn.Module` that you can use in any PyTorch pipeline.

The tokenizers in `transformers` are production implementations of the BPE algorithm we built from scratch in L10 notebook 01. They handle everything a real tokenizer needs to do: special tokens, batching, truncation, padding, and so on. The model classes are implemented in PyTorch and expose the full internal state (hidden representations, attention weights, intermediate activations), which makes them easy to inspect and modify.

We will look at two representative models: **DistilBERT** (an encoder in the BERT family, useful for classification) and **GPT-2 small** (a decoder in the GPT family, useful for generation). The two represent the two dominant architecture families for language, and they support different adaptation strategies.

In [ ]:
# Reference parameter counts for a few common models.
model_catalog = [
    ("DistilBERT-base",         "distilbert-base-uncased",   66),
    ("BERT-base",               "bert-base-uncased",         110),
    ("GPT-2 small",             "gpt2",                      124),
    ("GPT-2 medium",            "gpt2-medium",               345),
    ("RoBERTa-base",            "roberta-base",              125),
    ("T5-small",                "t5-small",                  60),
]
print(f"{'Model':<22} {'HF name':<26} {'Params (M)':>10}")
for nice, hf, p in model_catalog:
    print(f"{nice:<22} {hf:<26} {p:>10}")

## Demo 1: DistilBERT for Financial Sentiment Classification

**The task.** Classify short financial/economic headlines as expressing positive, negative, or neutral sentiment. This is a toy version of a problem that actually appears in applied finance: constructing sentiment indices from news text and using them as inputs to asset pricing or macroeconomic models.

**The dataset.** Rather than adding an external dataset dependency, we hand-construct a small labeled corpus inside the notebook. This keeps the example self-contained and makes it easy to see exactly what the model is being asked to do. The real `financial_phrasebank` dataset has thousands of examples; our toy version has about forty.

In [ ]:
# A small hand-labeled corpus of finance/economics headlines.
# Labels: 0=negative, 1=neutral, 2=positive
raw_data = [
    ("The company reported record quarterly earnings, beating analyst expectations.", 2),
    ("Revenue grew 30% year-over-year, driven by strong international sales.", 2),
    ("The central bank cut interest rates, boosting investor confidence.", 2),
    ("Unemployment fell to a ten-year low as hiring accelerated.", 2),
    ("The firm announced a major acquisition that analysts called strategic and well-priced.", 2),
    ("GDP growth exceeded forecasts for the third consecutive quarter.", 2),
    ("The new trade agreement is expected to increase exports substantially.", 2),
    ("Earnings per share rose sharply after successful cost-cutting measures.", 2),
    ("Consumer spending hit an all-time high during the holiday season.", 2),
    ("The firm's share price rallied on news of a dividend increase.", 2),
    ("The company issued a profit warning after weaker than expected sales.", 0),
    ("Shares plunged 15% after the board announced the CEO's resignation.", 0),
    ("The economy contracted for the second consecutive quarter, signaling recession.", 0),
    ("Inflation surged to a 40-year high, squeezing household budgets.", 0),
    ("The company filed for bankruptcy protection following years of losses.", 0),
    ("Exports collapsed as new tariffs took effect.", 0),
    ("The central bank raised rates unexpectedly, triggering a market sell-off.", 0),
    ("Revenue fell 20% year-over-year amid weak demand.", 0),
    ("The firm disclosed accounting irregularities, prompting an SEC investigation.", 0),
    ("Housing prices dropped sharply as mortgage rates climbed.", 0),
    ("The company released its quarterly results in line with expectations.", 1),
    ("The central bank left interest rates unchanged at its latest meeting.", 1),
    ("The firm announced the appointment of a new chief financial officer.", 1),
    ("Trading volume was in line with recent averages.", 1),
    ("The company updated its guidance but made no material changes.", 1),
    ("Inflation held steady at 2 percent, matching the central bank's target.", 1),
    ("The report noted continued uncertainty in emerging markets.", 1),
    ("Regulators announced a routine review of the firm's compliance program.", 1),
    ("The quarterly filing contained no surprises for investors.", 1),
    ("Management reiterated its full-year outlook without revision.", 1),
    ("The stock rose modestly on light trading volume.", 2),
    ("Factory orders declined more than analysts had forecast.", 0),
    ("Profit margins expanded as commodity prices fell.", 2),
    ("The merger was blocked by antitrust regulators.", 0),
    ("Retail sales grew in line with the long-run average.", 1),
    ("A new product launch exceeded initial sales forecasts.", 2),
    ("The firm cut its workforce by 10 percent to reduce costs.", 0),
    ("The pension fund met its return target for the year.", 1),
    ("Analysts downgraded the stock citing poor cash flow.", 0),
    ("The bond issuance was oversubscribed, reflecting strong investor demand.", 2),
]

rng = np.random.default_rng(0)
indices = rng.permutation(len(raw_data))
split = int(0.8 * len(raw_data))
train_pairs = [raw_data[i] for i in indices[:split]]
test_pairs = [raw_data[i] for i in indices[split:]]

print(f"Train size: {len(train_pairs)}")
print(f"Test size:  {len(test_pairs)}")

label_names = ["negative", "neutral", "positive"]
train_counts = np.bincount([y for _, y in train_pairs], minlength=3)
print("Train label counts:", dict(zip(label_names, train_counts.tolist())))

### Loading DistilBERT and Its Tokenizer

Three lines get us DistilBERT and its BPE tokenizer. We use the `AutoTokenizer` / `AutoModel` classes because they dispatch to the right specific class based on the checkpoint name — so the same code works for BERT, RoBERTa, DistilBERT, and most other encoder models.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
backbone = AutoModel.from_pretrained("distilbert-base-uncased").to(DEVICE).eval()

print(f"DistilBERT parameters: {sum(p.numel() for p in backbone.parameters()):,}")
print(f"Hidden size:           {backbone.config.hidden_size}")
print(f"Number of layers:      {backbone.config.num_hidden_layers}")

DistilBERT is a smaller distilled version of BERT-base: it has 6 transformer layers instead of 12, for about 66 million parameters instead of 110 million. On classification benchmarks it achieves most of BERT-base's accuracy at roughly half the compute. It is a good default for a classroom demo because it is fast and light while still behaving like a real BERT-family model.

Let's tokenize a sample sentence to see what the tokenizer produces.

In [ ]:
sample = "The central bank cut interest rates."
encoded = tokenizer(sample, return_tensors="pt")
print(f"Input text: {sample}")
print(f"Input IDs:  {encoded['input_ids'][0].tolist()}")
print(f"Tokens:     {tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])}")

Notice the special `[CLS]` and `[SEP]` tokens at the start and end. The `[CLS]` token is the one we use for classification — its final-layer hidden state is the "pooled" representation of the whole sentence. This convention was introduced in the original BERT paper: during pretraining BERT adds a learned embedding for `[CLS]` at the start of every input, and when the model is fine-tuned for classification, the classifier is attached to the `[CLS]` token's hidden state.

### Helper: Extract `[CLS]` Features

For the feature-extraction and linear-probe baselines we only need the frozen `[CLS]` features. Let's compute them once for the whole dataset — this is the only forward pass through DistilBERT we'll need for those two baselines.

In [ ]:
def extract_cls_features(model, tokenizer, texts, batch_size=16):
    """Return a tensor of [CLS] embeddings for each text."""
    model.eval()
    feats = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=64,
                            return_tensors="pt")
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            out = model(**enc)
            # DistilBERT's AutoModel returns last_hidden_state: (B, T, H)
            cls = out.last_hidden_state[:, 0, :]  # take the [CLS] token
            feats.append(cls.cpu())
    return torch.cat(feats, dim=0)


train_texts = [t for t, _ in train_pairs]
train_labels = torch.tensor([y for _, y in train_pairs])
test_texts = [t for t, _ in test_pairs]
test_labels = torch.tensor([y for _, y in test_pairs])

print("Extracting frozen DistilBERT [CLS] features...")
train_feats = extract_cls_features(backbone, tokenizer, train_texts)
test_feats = extract_cls_features(backbone, tokenizer, test_texts)
print(f"Train features: {train_feats.shape}  (H={train_feats.shape[1]})")
print(f"Test features:  {test_feats.shape}")

### Baseline 1: Feature Extraction + Scikit-Learn

The simplest baseline: fit a standard logistic regression on the frozen `[CLS]` features. Zero neural-network training, no GPU required, deterministic and fast. This is a strong default whenever your labeled dataset is tiny.

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(train_feats.numpy(), train_labels.numpy())
sk_acc = clf.score(test_feats.numpy(), test_labels.numpy())
print(f"sklearn logistic regression on frozen [CLS] features: test acc = {sk_acc:.3f}")

### Baseline 2: Linear Probe in PyTorch

The same idea, but with a `nn.Linear` trained via SGD inside PyTorch. This produces the same kind of model as the sklearn baseline, but fits cleanly into the PyTorch training loop we will use for full fine-tuning next. Small batch sizes and a few hundred epochs are fine because the feature tensor is already computed.

In [ ]:
torch.manual_seed(0)
probe = nn.Linear(backbone.config.hidden_size, 3).to(DEVICE)
opt = torch.optim.Adam(probe.parameters(), lr=3e-3)

train_feats_d = train_feats.to(DEVICE)
train_labels_d = train_labels.to(DEVICE)
test_feats_d = test_feats.to(DEVICE)
test_labels_d = test_labels.to(DEVICE)

for epoch in range(200):
    logits = probe(train_feats_d)
    loss = F.cross_entropy(logits, train_labels_d)
    opt.zero_grad()
    loss.backward()
    opt.step()

with torch.no_grad():
    probe_acc = (probe(test_feats_d).argmax(dim=1) == test_labels_d).float().mean().item()
probe_trainable = sum(p.numel() for p in probe.parameters())
print(f"PyTorch linear probe: test acc = {probe_acc:.3f}")
print(f"                       trainable params = {probe_trainable:,}")

### Baseline 3: Full Fine-Tuning

Now we let every parameter of DistilBERT receive gradient updates. The standard recipe is `AutoModelForSequenceClassification.from_pretrained(...)`, which loads the pretrained backbone and automatically adds a classification head on top. We fine-tune with a small learning rate (the standard BERT fine-tuning range is `2e-5` to `5e-5`).

We use a hand-written PyTorch training loop instead of the `Trainer` API so you can see exactly what is happening — no hidden magic.

In [ ]:
torch.manual_seed(0)
classifier = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3
).to(DEVICE)

# Tokenize once and keep tensors on device.
train_enc = tokenizer(train_texts, padding=True, truncation=True, max_length=64,
                       return_tensors="pt")
test_enc = tokenizer(test_texts, padding=True, truncation=True, max_length=64,
                      return_tensors="pt")
train_enc = {k: v.to(DEVICE) for k, v in train_enc.items()}
test_enc = {k: v.to(DEVICE) for k, v in test_enc.items()}
train_labels_d = train_labels.to(DEVICE)
test_labels_d = test_labels.to(DEVICE)


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"Full fine-tune trainable params: {count_trainable(classifier):,}")

opt = torch.optim.AdamW(classifier.parameters(), lr=3e-5)

t0 = time.time()
for epoch in range(4):
    classifier.train()
    # Single-batch training (dataset is small)
    out = classifier(**train_enc, labels=train_labels_d)
    loss = out.loss
    opt.zero_grad()
    loss.backward()
    opt.step()
    print(f"  epoch {epoch + 1} loss {loss.item():.3f}")
full_time = time.time() - t0

classifier.eval()
with torch.no_grad():
    preds = classifier(**test_enc).logits.argmax(dim=1)
    full_acc = (preds == test_labels_d).float().mean().item()

print(f"\nFull fine-tune test acc = {full_acc:.3f}  ({full_time:.1f}s)")

In [ ]:
# Three-way comparison
rows = [
    ("sklearn LogReg (feats)", sk_acc,   0),
    ("PyTorch linear probe",   probe_acc, probe_trainable),
    ("Full fine-tune",         full_acc,  count_trainable(classifier)),
]

print(f"{'Strategy':<25} {'Test acc':>10}  {'Trainable (neural)':>22}")
print("-" * 60)
for name, acc, params in rows:
    print(f"{name:<25} {acc:>10.3f}  {params:>22,}")

On a dataset this small the three strategies usually produce similar accuracy — the pretrained features are strong enough that the choice of classifier barely matters. The point is the parameter count: the linear probe trains about 2,300 parameters, while full fine-tuning updates 66 million. Exact accuracy numbers will fluctuate across seeds, but the rankings are usually preserved.

### Visualizing Attention

One of the nice things about transformer-family models is that you can literally open them up and look at where the model is attending. Let's ask a fine-tuned DistilBERT to classify a sentence and then visualize the attention weights from the last layer's `[CLS]` token to every other token.

In [ ]:
# Reload with attention outputs enabled
classifier.config.output_attentions = True
classifier.eval()

sample_text = "The firm announced a major acquisition that analysts called strategic."
enc = tokenizer(sample_text, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    out = classifier(**enc, output_attentions=True)
    pred = out.logits.argmax(dim=1).item()

# Attention shape: (layers, batch, heads, T, T)
attentions = torch.stack(out.attentions)  # (L, B, H, T, T)
# Average over heads in the last layer, look at row 0 ([CLS]'s attention over all tokens)
cls_attn = attentions[-1, 0].mean(dim=0)[0].cpu().numpy()  # (T,)

tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

fig, ax = plt.subplots(figsize=(12, 2))
ax.imshow(cls_attn[None, :], aspect="auto", cmap="viridis")
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
ax.set_yticks([])
ax.set_title(f"[CLS] attention (last layer, averaged over heads)\nprediction: {label_names[pred]}")
plt.tight_layout()
plt.show()

print(f"Top-3 attended tokens:")
top3 = np.argsort(cls_attn)[::-1][:3]
for i in top3:
    print(f"  {tokens[i]:<15} weight={cls_attn[i]:.3f}")

The attention weights show which parts of the sentence the classifier is looking at when making its decision. Of course, attention is not the same as "explanation" in a rigorous sense — there is a literature on whether attention weights should be interpreted as importance scores — but they do give a useful qualitative picture of what the model is doing.

## Demo 2: GPT-2 + Hand-Rolled LoRA for Domain Adaptation

Now we switch from classification to generation, and from encoder to decoder. The task: adapt GPT-2 small so that it generates text in the style of Adam Smith's *The Wealth of Nations* — the same corpus we used in L10 notebook 01. This is called **domain adaptation**.

We will do this using **LoRA** — the low-rank adaptation scheme we derived in notebook 01. Instead of fine-tuning all 124 million parameters of GPT-2, we freeze the base model completely and insert a small number of trainable low-rank matrices into the attention layers. The parameter count drops by roughly 100×.

The twist is that GPT-2's attention layers in Hugging Face are not implemented as `nn.Linear`. They use a custom class called `Conv1D` which is functionally equivalent but has a transposed weight layout. We will need to write our LoRA wrapper slightly differently to handle this — a nice real-world example of the practical friction you run into when doing this kind of surgery on production models.

### Load GPT-2 and Generate Before Fine-Tuning

Let's start by loading GPT-2 and generating some text from a finance-related prompt. This is the "before" state against which we will compare the fine-tuned model.

In [ ]:
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token  # GPT-2 has no explicit pad token
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
gpt2.eval()

print(f"GPT-2 parameters: {sum(p.numel() for p in gpt2.parameters()):,}")
print(f"Hidden size:       {gpt2.config.n_embd}")
print(f"Layers:            {gpt2.config.n_layer}")
print(f"Heads:             {gpt2.config.n_head}")

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=60, temperature=0.8, seed=0):
    torch.manual_seed(seed)
    ids = gpt2_tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    out = model.generate(
        ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_k=50,
        pad_token_id=gpt2_tokenizer.eos_token_id,
    )
    return gpt2_tokenizer.decode(out[0], skip_special_tokens=True)


PROMPT = "The division of labour in the manufacture of pins"
print("=== Base GPT-2 (pretrained, no fine-tuning) ===")
print(generate(gpt2, PROMPT))

The output is generic modern English — GPT-2 is trained on WebText, which is dominated by contemporary internet writing. It has never been specifically trained on 18th-century prose.

### The Training Corpus

We reuse Adam Smith's text from L10 notebook 01. In real fine-tuning you would use a much larger corpus, but even this tiny one will visibly shift the model's output style after a few hundred training steps.

In [ ]:
smith_corpus = (
    "The annual labour of every nation is the fund which originally supplies "
    "it with all the necessaries and conveniences of life which it annually "
    "consumes, and which consist either in the immediate produce of that "
    "labour, or in what is purchased with that produce from other nations. "
    "According therefore as this produce, or what is purchased with it, bears "
    "a greater or smaller proportion to the number of those who are to consume "
    "it, the nation will be better or worse supplied with all the necessaries "
    "and conveniences for which it has occasion. But this proportion must in "
    "every nation be regulated by two different circumstances; first, by the "
    "skill, dexterity, and judgment with which its labour is generally applied; "
    "and, secondly, by the proportion between the number of those who are "
    "employed in useful labour, and that of those who are not so employed. "
    "The greatest improvement in the productive powers of labour, and the "
    "greater part of the skill, dexterity, and judgment with which it is "
    "anywhere directed, or applied, seem to have been the effects of the "
    "division of labour. The effects of the division of labour, in the general "
    "business of society, will be more easily understood by considering in "
    "what manner it operates in some particular manufactures. "
    "To take an example, therefore, from a very trifling manufacture; but one "
    "in which the division of labour has been very often taken notice of, the "
    "trade of the pin-maker; a workman not educated to this business, nor "
    "acquainted with the use of the machinery employed in it, could scarce, "
    "perhaps, with his utmost industry, make one pin in a day, and certainly "
    "could not make twenty. But in the way in which this business is now "
    "carried on, not only the whole work is a peculiar trade, but it is "
    "divided into a number of branches, of which the greater part are likewise "
    "a peculiar trade. One man draws out the wire, another straights it, a "
    "third cuts it, a fourth points it, a fifth grinds it at the top for "
    "receiving the head; to make the head requires two or three distinct "
    "operations; to put it on is a peculiar business, to whiten the pins is "
    "another; it is even a trade by itself to put them into the paper; and the "
    "important business of making a pin is, in this manner, divided into about "
    "eighteen distinct operations."
)

# Tokenize the whole corpus as a single long sequence
full_ids = gpt2_tokenizer(smith_corpus, return_tensors="pt").input_ids[0]
print(f"Corpus length in tokens: {len(full_ids)}")

### Implementing LoRA for GPT-2's `Conv1D`

In notebook 01 we implemented `LoRALinear` that wraps an `nn.Linear`. Let's check what GPT-2 uses for its attention projections.

In [ ]:
# Peek at one attention block
block = gpt2.transformer.h[0]
print(block.attn)
print()
print(f"c_attn weight shape: {block.attn.c_attn.weight.shape}  (nx, nf)")
print(f"c_proj weight shape: {block.attn.c_proj.weight.shape}  (nx, nf)")

GPT-2 uses `transformers.pytorch_utils.Conv1D` instead of `nn.Linear`. The two are functionally equivalent but have different weight layouts:

- `nn.Linear` has weight shape `(out, in)` and computes `y = x @ W.T + b`.
- `Conv1D` has weight shape `(in, out)` and computes `y = x @ W + b`.

This is purely a transposition difference — the math is the same — but it means we cannot use our `LoRALinear` from notebook 01 directly. We need a small `LoRAConv1D` wrapper that matches the `Conv1D` forward pass.

The derivation is the same as in notebook 01: we want to add a low-rank correction $BA$ to the weight, so the forward pass becomes

$$y = x W + b + \frac{\alpha}{r} x A B$$

where $A \in \mathbb{R}^{n_x \times r}$ and $B \in \mathbb{R}^{r \times n_f}$. (Compare to notebook 01, where the roles of $A$ and $B$ are shaped for the `nn.Linear` convention.) Initialization is $A \sim \mathcal{N}$, $B = 0$, so the initial forward is exactly the frozen base.

In [ ]:
class LoRAConv1D(nn.Module):
    """LoRA wrapper around a Hugging Face ``Conv1D`` layer.

    ``Conv1D`` has weight shape ``(nx, nf)`` and computes ``x @ W + b``.
    We add a trainable low-rank correction ``(alpha / r) * x @ A @ B``,
    where ``A`` has shape ``(nx, r)`` and ``B`` has shape ``(r, nf)``.
    ``A`` is initialized to small random values and ``B`` to zero, so the
    initial forward pass is exactly equal to the frozen base layer.
    """

    def __init__(self, base: Conv1D, r: int = 4, alpha: float = 8.0):
        super().__init__()
        nx, nf = base.weight.shape
        self.nx = nx
        self.nf = nf
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False

        self.lora_A = nn.Parameter(torch.zeros(nx, r))
        self.lora_B = nn.Parameter(torch.zeros(r, nf))
        nn.init.kaiming_uniform_(self.lora_A, a=5 ** 0.5)
        # B stays at zero so the initial output matches the base.

    def forward(self, x):
        base_out = self.base(x)
        lora_out = (x @ self.lora_A) @ self.lora_B
        return base_out + self.scaling * lora_out


# Sanity check: at initialization the LoRA-wrapped layer must equal the base.
original = copy.deepcopy(gpt2.transformer.h[0].attn.c_attn)
wrapped = LoRAConv1D(original, r=4, alpha=8.0)

x_test = torch.randn(2, 5, original.weight.shape[0])
with torch.no_grad():
    a = original(x_test)
    b = wrapped(x_test)
assert torch.allclose(a, b, atol=1e-6), "LoRA init should match base"
print("LoRAConv1D init matches base — OK.")

### Applying LoRA to Every Attention Projection

Now we walk through every transformer block in GPT-2 and replace the attention projections (`c_attn` and `c_proj`) with LoRA-wrapped versions. We leave the MLP layers alone — a common LoRA practice that saves parameters without losing much capability, since the attention projections are typically where the bulk of adaptation happens.

In [ ]:
# Reload GPT-2 fresh so we don't accumulate modifications across cells.
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)

# Freeze every parameter first.
for p in gpt2.parameters():
    p.requires_grad = False

# Walk the transformer blocks and wrap the attention Conv1Ds.
LORA_R = 4
LORA_ALPHA = 8.0

num_wrapped = 0
for block in gpt2.transformer.h:
    block.attn.c_attn = LoRAConv1D(block.attn.c_attn, r=LORA_R, alpha=LORA_ALPHA)
    block.attn.c_proj = LoRAConv1D(block.attn.c_proj, r=LORA_R, alpha=LORA_ALPHA)
    num_wrapped += 2

# Move any newly-created parameters to the device.
gpt2 = gpt2.to(DEVICE)

trainable = [(n, p) for n, p in gpt2.named_parameters() if p.requires_grad]
total_trainable = sum(p.numel() for _, p in trainable)
total_params = sum(p.numel() for p in gpt2.parameters())

print(f"LoRA layers inserted:  {num_wrapped}")
print(f"Trainable parameters:  {total_trainable:,}")
print(f"Total parameters:      {total_params:,}")
print(f"Trainable fraction:    {100 * total_trainable / total_params:.3f}%")
print(f"\nFirst few trainable parameter names:")
for n, _ in trainable[:4]:
    print(f"  {n}")

We wrapped 24 layers (2 per block, 12 blocks) and ended up with under 1% of the model as trainable parameters. Fine-tuning this will be much lighter than full fine-tuning — memory, compute, and final adapter size all shrink dramatically.

### Training the LoRA Adapters

Next-token prediction loss on the Wealth of Nations corpus, a handful of epochs. We form training batches by sliding a fixed-size window over the tokenized corpus.

In [ ]:
def make_chunks(ids, block_size=64):
    """Split a long sequence of token IDs into fixed-size training chunks."""
    chunks = []
    for i in range(0, len(ids) - block_size, block_size):
        chunks.append(ids[i : i + block_size + 1])
    return torch.stack(chunks)


train_chunks = make_chunks(full_ids, block_size=64).to(DEVICE)
print(f"Training chunks: {train_chunks.shape}  (n_chunks, block_size+1)")

# Build inputs and targets: input is tokens[:-1], target is tokens[1:]
input_ids = train_chunks[:, :-1]
target_ids = train_chunks[:, 1:]

opt = torch.optim.AdamW(
    [p for p in gpt2.parameters() if p.requires_grad], lr=5e-4
)

gpt2.train()
t0 = time.time()
for epoch in range(20):
    # Full-batch — dataset is small enough
    out = gpt2(input_ids=input_ids)
    logits = out.logits  # (B, T, V)
    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target_ids.reshape(-1),
    )
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (epoch + 1) % 5 == 0:
        print(f"  epoch {epoch + 1:3d}  loss {loss.item():.3f}")

print(f"\nTraining time: {time.time() - t0:.1f}s")

### Generate After Fine-Tuning

The same prompt as before, now through the LoRA-adapted model.

In [ ]:
gpt2.eval()
print("=== GPT-2 after LoRA fine-tuning on Wealth of Nations ===")
print(generate(gpt2, PROMPT))
print()
print("=== Another generation ===")
print(generate(gpt2, "The annual labour of every nation", seed=1))

The output has shifted in the direction of 18th-century prose: longer, more formal sentences, vocabulary like *labour*, *dexterity*, *manufacture*, and the kinds of clause structures Smith uses. It is not perfect — we only trained for 20 epochs on a few hundred tokens — but the *style* is visibly different from the baseline GPT-2 output. This is exactly the kind of domain adaptation LoRA excels at: a small number of trainable parameters shift the model's behavior in a specific direction without disturbing the underlying language competence.

### LoRA vs. Full Fine-Tuning: Memory and Parameter Accounting

Let's make the comparison between LoRA and hypothetical full fine-tuning explicit.

In [ ]:
# For comparison, count what full fine-tuning would train.
fresh_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
full_ft_params = sum(p.numel() for p in fresh_gpt2.parameters())

lora_params = total_trainable

print(f"{'Strategy':<25} {'Trainable':>15}  {'Fraction of total':>22}")
print("-" * 66)
print(f"{'Full fine-tuning':<25} {full_ft_params:>15,}  {100:>21.3f}%")
print(f"{'LoRA (r=4, attn only)':<25} {lora_params:>15,}  {100 * lora_params / full_ft_params:>21.3f}%")
print(f"\nRatio: LoRA uses {full_ft_params / lora_params:,.0f}x fewer trainable parameters.")

LoRA trains roughly a thousand times fewer parameters than full fine-tuning. Three practical consequences follow:

1. **Memory.** Training memory is dominated by optimizer state (Adam stores two moment estimates per trainable parameter) and gradients. Training 0.1% of the parameters means ~1/1000 the optimizer state, which lets very large models fit on a single GPU.

2. **Storage.** A trained LoRA adapter for GPT-2 is a few megabytes, versus ~500 MB for a full fine-tune checkpoint. This makes it practical to keep many task-specific adapters on disk and load them on demand.

3. **Adapter swapping.** Because each adapter is small and additive, you can train multiple LoRA adapters for different tasks and load them into the same base model at inference time. Some frameworks even support merging multiple adapters in a single pass.

These benefits are the reason LoRA has become the default adaptation method for very large generative models. For GPT-2 small the absolute savings are modest; for Llama-70B they are the difference between "impossible on consumer hardware" and "feasible on a single A100."

## The Broader Language Foundation Model Landscape

| Family | Architecture | Canonical models | Typical adaptation |
|---|---|---|---|
| **Encoder** | Bidirectional transformer, trained with masked language modeling (MLM) | BERT, RoBERTa, DistilBERT | Fine-tune for classification, NER, embedding |
| **Decoder** | Causal transformer, trained with next-token prediction | GPT-2, GPT-3, Llama, Mistral, Qwen, Claude | Fine-tune (often LoRA) for generation; prompt/ICL for everything else |
| **Encoder-decoder** | Separate encoder and decoder, trained with sequence-to-sequence objectives | T5, BART, FLAN-T5 | Fine-tune for translation, summarization |

**Encoder models (BERT family).** Good at producing dense representations of input text. The natural adaptation pattern is the one we used in Demo 1: take the final `[CLS]` token embedding and attach a classification head. Encoder models are still very competitive for classification, entity extraction, and retrieval tasks where you need a fixed-dimensional representation of a text span. They are also much cheaper than decoder models at inference.

**Decoder models (GPT family).** The dominant architecture for modern LLMs. Trained purely on next-token prediction, they are powerful generators and, somewhat surprisingly, can also do classification and extraction very well through prompting. The modern frontier (GPT-4, Claude, Llama 3) consists of large decoder-only models post-trained with the supervised fine-tuning and preference optimization techniques from L10 notebook 03. For these models, the three common adaptation patterns are prompting, LoRA fine-tuning, and RAG.

**Encoder-decoder models (T5 family).** A middle ground with separate encoder and decoder stacks, originally popularized for translation and summarization. Less common than pure decoder models in 2024–2025, but still widely used for sequence-to-sequence tasks where you want a distinct encoding of the input.

### Base vs. Instruction-Tuned Models

Most modern language foundation models come in two flavors:

- **Base model** — trained only on next-token prediction over a large corpus. Good at predicting what text naturally follows the prompt, but does not follow instructions.
- **Instruct / Chat model** — the base model further trained with supervised fine-tuning and preference optimization (SFT + RLHF/DPO, from L10 notebook 03). Follows instructions, has a chat format, behaves like an assistant.

When you fine-tune a foundation model for a specific task, the right starting point depends on what you want:

- **Starting from the base model** gives you maximum flexibility but requires you to teach the model task behavior from scratch.
- **Starting from an instruct model** preserves instruction-following behavior and lets you make more targeted changes. This is usually the right choice for downstream applications where you want the model to remain helpful and polite.

## When to Fine-Tune vs. Prompt vs. RAG

Fine-tuning is only one of several ways to adapt a language model. The other two are **prompting** (and in-context learning) and **retrieval-augmented generation (RAG)**. Each has a different cost structure.

| Approach | Fixed cost | Marginal cost per query | Good for |
|---|---|---|---|
| **Prompting / ICL** | None | Higher — prompt tokens repeated on every call | Any task, quick prototyping, small volumes |
| **RAG** | Build a retrieval index | Moderate — retrieval + augmented prompt | Knowledge-grounded tasks, fresh information |
| **Fine-tuning (full or LoRA)** | Training compute + data collection | Low — short prompts, cached weights | High-volume tasks, specialized behaviors, niche styles |

For economists, the fixed-versus-marginal-cost framing is familiar. If you will query the model a few thousand times a month, prompting is cheaper. If you will query it billions of times per month, fine-tuning amortizes quickly. If you need the model to reason over facts that change frequently (company financials, regulatory filings), RAG is usually better than fine-tuning because you can update the knowledge base without retraining. And if the task is about *style* or *format* rather than facts, fine-tuning (especially LoRA) is usually the most effective option.

In practice, most production systems use a combination: prompt engineering for the outer behavior, LoRA or full fine-tuning for domain-specific skills, and RAG for facts.

## Summary

- Hugging Face's `transformers` library is the standard way to load pretrained language models in three lines. The core classes are `AutoTokenizer`, `AutoModel`, and task-specific variants like `AutoModelForSequenceClassification` and `AutoModelForCausalLM`.
- **Encoder models** (BERT family) are natural for classification: attach a classifier to the `[CLS]` token representation and fine-tune with a hand-written PyTorch loop.
- On a tiny labeled dataset, feature extraction and linear probes are strong baselines that rival full fine-tuning at a fraction of the parameter count.
- **Decoder models** (GPT family) are natural for generation. GPT-2's attention projections use `Conv1D` rather than `nn.Linear`, so our notebook 01 `LoRALinear` needed a sibling `LoRAConv1D` with a transposed weight layout.
- **Hand-implementing LoRA** and applying it to GPT-2 gave us a working domain-adaptation pipeline that trains under 1% of the model's parameters.
- The broader landscape includes encoder, decoder, and encoder-decoder families; instruction-tuned models; and alternative adaptation strategies (prompting, RAG) that are often better than fine-tuning for common use cases.

This concludes the L11 foundation models lecture. You now know how practitioners actually use deep learning models — not by training from scratch, but by taking someone else's pretrained model and adapting it as cheaply as possible to the task at hand.

## References

- Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *NAACL 2019*.
- Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter. *arXiv:1910.01108*.
- Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language models are unsupervised multitask learners. *OpenAI Technical Report*.
- Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L., & Chen, W. (2021). LoRA: Low-rank adaptation of large language models. *arXiv:2106.09685*.
- Wolf, T., Debut, L., Sanh, V., et al. (2020). Transformers: State-of-the-art natural language processing. *EMNLP 2020: System Demonstrations*.
- Raffel, C., Shazeer, N., Roberts, A., et al. (2020). Exploring the limits of transfer learning with a unified text-to-text transformer (T5). *JMLR*.
- Lewis, P., Perez, E., Piktus, A., et al. (2020). Retrieval-augmented generation for knowledge-intensive NLP tasks. *NeurIPS 2020*.
